# Forecasting Tournament R2 Presentation Figure

Standalone presentation figure for the PFE defense slide: Demand Forecasting - Evaluation.

This notebook only loads saved evaluation metrics and redraws the R2 leaderboard. It does not retrain models, edit official report figures, or modify model artifacts.


In [ ]:

from pathlib import Path
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / 'notebooks_experiments'
FINAL_DIR = ROOT / 'backend' / 'Ai models' / 'data' / 'final' / 'Module 2'
NOTEBOOK_DIR = ROOT / 'backend' / 'Ai models' / 'notebooks' / 'Module 2'
REPORT_PDF = ROOT / 'rapport (1).pdf'

SOURCES = {
    'statistical': FINAL_DIR / 'statistical_comparison_weekly.json',
    'ml': FINAL_DIR / 'ml_comparison_summary.json',
    'lstm': FINAL_DIR / 'lstm_summary_weekly.json',
    'tft_notebook': NOTEBOOK_DIR / '06b_tft_weekly_aligned.ipynb',
    'report': REPORT_PDF,
}

for label, path in SOURCES.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing required source for {label}: {path}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

statistical = load_json(SOURCES['statistical'])
ml = load_json(SOURCES['ml'])
lstm = load_json(SOURCES['lstm'])

records = []

def add_from_summary(model, family, role, source_label, source_path, key, summary):
    overall = summary['overall_metrics'][key]
    records.append({
        'model': model,
        'family': family,
        'expected_role': role,
        'source_file': str(source_path.relative_to(ROOT)).replace('\\', '/'),
        'val_wape': float(overall['val']['wape']),
        'test_wape': float(overall['test']['wape']),
        'val_r2': float(overall['val']['r2']),
        'test_r2': float(overall['test']['r2']),
        'metric_source_note': source_label,
    })

add_from_summary('Prophet', 'Statistical baseline', 'Baseline', 'final JSON summary', SOURCES['statistical'], 'prophet', statistical)
add_from_summary('SARIMA', 'Statistical baseline', 'Baseline', 'final JSON summary', SOURCES['statistical'], 'sarima', statistical)
add_from_summary('XGBoost', 'Machine learning', 'Strong challenger', 'final JSON summary', SOURCES['ml'], 'xgboost', ml)
add_from_summary('LightGBM', 'Machine learning', 'Selected model', 'final JSON summary', SOURCES['ml'], 'lightgbm', ml)
add_from_summary('LSTM', 'Deep learning', 'Deep learning benchmark', 'final JSON summary', SOURCES['lstm'], 'lstm', {'overall_metrics': {'lstm': lstm['overall_metrics']}})

# TFT standalone final metrics file is absent; these report values are also present in the executed TFT notebook output.
records.append({
    'model': 'TFT',
    'family': 'Deep learning',
    'expected_role': 'Advanced benchmark',
    'source_file': 'rapport (1).pdf Tables 6.10-6.11; backend/Ai models/notebooks/Module 2/06b_tft_weekly_aligned.ipynb output',
    'val_wape': 0.3471,
    'test_wape': 0.2739,
    'val_r2': 0.5475,
    'test_r2': 0.6362,
    'metric_source_note': 'report table plus executed TFT notebook output; no standalone final TFT summary file found',
})

metrics = pd.DataFrame(records)
metrics['selected_model'] = metrics['model'].eq('LightGBM')
metrics = metrics.sort_values('val_r2', ascending=False).reset_index(drop=True)
metrics['rank'] = np.arange(1, len(metrics) + 1)
metrics = metrics[['rank', 'model', 'family', 'expected_role', 'selected_model', 'val_r2', 'test_r2', 'val_wape', 'test_wape', 'source_file', 'metric_source_note']]

REPORT_EXPECTED = {
    'Prophet': {'val_r2': 0.4937, 'test_r2': 0.6196, 'val_wape': 0.3734, 'test_wape': 0.2837},
    'SARIMA': {'val_r2': 0.4211, 'test_r2': 0.5821, 'val_wape': 0.4019, 'test_wape': 0.2867},
    'XGBoost': {'val_r2': 0.6579, 'test_r2': 0.6707, 'val_wape': 0.3073, 'test_wape': 0.2614},
    'LightGBM': {'val_r2': 0.6614, 'test_r2': 0.6681, 'val_wape': 0.3061, 'test_wape': 0.2617},
    'LSTM': {'val_r2': 0.6165, 'test_r2': 0.6401, 'val_wape': 0.3175, 'test_wape': 0.2676},
    'TFT': {'val_r2': 0.5475, 'test_r2': 0.6362, 'val_wape': 0.3471, 'test_wape': 0.2739},
}

for _, row in metrics.iterrows():
    expected = REPORT_EXPECTED[row['model']]
    for col, expected_value in expected.items():
        actual = round(float(row[col]), 4)
        if actual != expected_value:
            raise AssertionError(f"{row['model']} {col} mismatch: figure metric={actual}, report={expected_value}")

ranking = metrics[['rank', 'model', 'val_r2', 'test_r2', 'val_wape', 'test_wape', 'expected_role']]
print('Data integrity checks')
print('- Ranking metric: validation R2 (higher is better).')
print('- Evaluation protocol: weekly item-demand forecasting with validation/test splits used in the Module 2 evaluation notebooks and report Section 6.3.')
print(f'- Models ranked: {", ".join(metrics["model"].tolist())}.')
print('- Selected model according to report: LightGBM.')
print('- Figure metrics match: validation R2, test R2, validation WAPE, and test WAPE match rapport/report tables at displayed precision.')
print('\nLeaderboard used for the figure:')
print(ranking.to_string(index=False, formatters={'val_r2': '{:.4f}'.format, 'test_r2': '{:.4f}'.format, 'val_wape': '{:.4f}'.format, 'test_wape': '{:.4f}'.format}))

csv_path = OUT_DIR / 'forecasting_tournament_r2_metrics_used.csv'
metrics.to_csv(csv_path, index=False)

notes_path = OUT_DIR / 'forecasting_tournament_r2_figure_notes.md'
notes = f'''# Forecasting Tournament R2 Figure Notes

Generated by: `notebooks_experiments/forecasting_tournament_r2_presentation_figure.ipynb`

## Purpose

Standalone PFE defense slide figure for Demand Forecasting - Evaluation, focused on R2 instead of WAPE. The figure is styled as a conventional Matplotlib evaluation plot.

## Ranking Metric

The figure ranks models by validation R2, higher is better. This follows the report explanation that LightGBM has the best validation WAPE and validation R2, while XGBoost is the closest challenger on the test set.

## Exact Ranking Used

{ranking.to_markdown(index=False, floatfmt='.4f')}

## Sources

- `backend/Ai models/data/final/Module 2/statistical_comparison_weekly.json` for Prophet and SARIMA.
- `backend/Ai models/data/final/Module 2/ml_comparison_summary.json` for XGBoost and LightGBM.
- `backend/Ai models/data/final/Module 2/lstm_summary_weekly.json` for LSTM.
- `rapport (1).pdf` Tables 6.10-6.11 and `backend/Ai models/notebooks/Module 2/06b_tft_weekly_aligned.ipynb` executed output for TFT.

## Integrity Notes

- The plotted/displayed validation R2 and test R2 values match the report tables at displayed precision.
- WAPE values are also checked to preserve consistency with the report decision logic.
- No training, retraining, model artifact update, report figure edit, or evaluation-output overwrite was performed.

## Output Files

- `notebooks_experiments/figure_forecasting_tournament_r2.png`
- `notebooks_experiments/figure_forecasting_tournament_r2_highres.png`
- `notebooks_experiments/forecasting_tournament_r2_metrics_used.csv`
'''
notes_path.write_text(notes, encoding='utf-8')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})

plot_df = metrics.copy()
x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(12, 6.75), facecolor='white')
ax.set_facecolor('white')

val_bars = ax.bar(
    x - width / 2,
    plot_df['val_r2'],
    width,
    label='Validation R2',
    color='#4C78A8',
    edgecolor='black',
    linewidth=0.6,
)
test_bars = ax.bar(
    x + width / 2,
    plot_df['test_r2'],
    width,
    label='Test R2',
    color='#F58518',
    edgecolor='black',
    linewidth=0.6,
)


for bars in (val_bars, test_bars):
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8, rotation=90)


ax.set_ylabel('R2')
ax.set_xlabel('Model')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['model'])
ax.set_ylim(0.35, 0.76)
ax.legend(loc='upper right', frameon=True)
ax.grid(axis='y', color='#d9d9d9', linewidth=0.8)
ax.grid(axis='x', visible=False)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

fig.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])

png_path = OUT_DIR / 'figure_forecasting_tournament_r2.png'
highres_path = OUT_DIR / 'figure_forecasting_tournament_r2_highres.png'
fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
fig.savefig(highres_path, dpi=320, bbox_inches='tight', facecolor='white')
plt.close(fig)

print(f'Wrote {png_path.relative_to(ROOT)}')
print(f'Wrote {highres_path.relative_to(ROOT)}')
print(f'Wrote {csv_path.relative_to(ROOT)}')
print(f'Wrote {notes_path.relative_to(ROOT)}')
